In [1]:
import os
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 1000)  # 한 줄에 출력할 가로 폭을 넓게 설정


hyd_tags = os.path.join("..", "Data", "03-01_유압·열설비_신호_계통태그목록.csv")
hyd_df = os.path.join("..", "Data", "03-01_유압·열설비_신호_유압운전.csv")

hyd_tags = pd.read_csv(hyd_tags, encoding="utf-8")
hyd_df = pd.read_csv(hyd_df, encoding="utf-8")

print(hyd_df.head())
print()
print(hyd_tags.columns)

         date  HYD01_PRESS_PUMP  HYD01_PRESS_FILT_IN  HYD01_PRESS_FILT_OUT  HYD01_FLOW  HYD01_OILTEMP  HYD01_LEVEL  HYD01_PUMP_CURRENT
0  2026-01-01             152.0                150.0                 147.5       118.0           42.0         88.0                31.5
1  2026-01-02             153.0                151.0                 148.5       118.5           42.0         88.0                31.5
2  2026-01-03             152.0                150.0                 147.5       117.5           42.5         88.0                31.0
3  2026-01-04             151.0                149.0                 146.0       118.0           42.5         88.0                32.0
4  2026-01-05             153.0                151.0                 148.0       118.5           42.5         88.0                31.5

Index(['tag', 'system', 'physical_qty', 'unit', 'reference', 'circuit_position', 'sampling_sec'], dtype='str')


In [4]:
print(hyd_tags.loc[hyd_tags["tag"].str.startswith("HYD"), ["physical_qty","unit","reference","circuit_position"]])




  physical_qty   unit reference circuit_position
0           압력    bar      게이지압           펌프 토출부
1           압력    bar      게이지압            필터 전단
2           압력    bar      게이지압            필터 후단
3           차압    bar        차압       필터 전후단 계산값
4           유량  L/min      해당없음         펌프 토출 배관
5           온도   degC      해당없음            탱크 내부
6           유면      %      해당없음           탱크 유면계
7           전류      A      해당없음           펌프 제어반
8           개도      %      해당없음      방향 제어 밸브 지령
9           개도      %      해당없음      방향 제어 밸브 실제


In [7]:
COL = [
    "HYD01_PRESS_PUMP",
    "HYD01_FLOW",
    "HYD01_OILTEMP",
    "HYD01_LEVEL",
    "HYD01_PUMP_CURRENT",
]

normal_30 = hyd_df.head(30)
recent_10 = hyd_df.tail(10)

# 처음 30일
print(normal_30.head(30)[COL].agg(["mean","min","max"]).round(2))

# 최근 10일
print(recent_10.head(30)[COL].agg(["mean", "min", "max"]).round(2))

      HYD01_PRESS_PUMP  HYD01_FLOW  HYD01_OILTEMP  HYD01_LEVEL  HYD01_PUMP_CURRENT
mean             152.3      117.95          42.35         88.0               31.45
min              151.0      117.00          42.00         88.0               31.00
max              154.0      118.50          43.00         88.0               32.00
      HYD01_PRESS_PUMP  HYD01_FLOW  HYD01_OILTEMP  HYD01_LEVEL  HYD01_PUMP_CURRENT
mean             149.5      116.95          48.35         88.0               32.45
min              148.0      116.00          47.50         88.0               32.00
max              151.0      117.50          49.50         88.0               33.00


In [10]:
# 차압이란? 필터 전단 압력 - 필터 후단 압력
hyd_df["DP"] = (hyd_df["HYD01_PRESS_FILT_IN"] - hyd_df["HYD01_PRESS_FILT_OUT"]).round(2)

print(hyd_df["DP"].head(20))

0     2.5
1     2.5
2     2.5
3     3.0
4     3.0
5     3.0
6     3.0
7     3.0
8     3.0
9     3.5
10    3.5
11    3.5
12    3.5
13    3.5
14    3.5
15    4.0
16    4.0
17    4.0
18    4.0
19    4.0
Name: DP, dtype: float64


In [17]:
# 1도 가져오고 싶고 30도 가져오고 싶으니, lo와 hi 두개로 순환.
for lo, hi in [(1,30),(31,60),(61,90)]:
    # 30일과 60일에 필터를 교체했다고 가정하자.
    seg = hyd_df.iloc[lo-1:hi]
    print(
        lo,
        hi,
        seg["DP"].iloc[0],
        seg["DP"].iloc[-1],
        round(seg["DP"].iloc[-1] - seg["DP"].iloc[0], 2 ),
    )

1 30 2.5 5.0 2.5
31 60 2.5 6.0 3.5
61 90 2.5 7.0 4.5
